In [ ]:
pip install confluent_kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 48.0 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
from confluent_kafka import Consumer
from sklearn.ensemble import IsolationForest

conf = {
    'bootstrap.servers': 'pkc-921jm.us-east-2.aws.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': 'NTT5JF77OADZWJPK',
    'sasl.password': 'cfltbntz/OnhwkK6SRHD3/lrYlceiXau+yn3J+L9K0r3WPu8acgUkzU0fmkv1ZDQ',
    'group.id': 'transactions',
    'auto.offset.reset': 'latest'
}

consumer = Consumer(conf)
consumer.subscribe(['stream_demo'])

print("Consumer with ML started...")

window = []
model = IsolationForest(contamination=0.05)

while True:
    msg = consumer.poll(1.0)

    if msg is None:
        continue
    if msg.error():
        print("Error:", msg.error())
        continue

    data = json.loads(msg.value().decode('utf-8'))
    amount = data["amount"]

    # collect data
    window.append([amount])

    # train after enough data
    if len(window) > 20:
        model.fit(window)

        pred = model.predict([[amount]])

        if pred[0] == -1:
            print("ML ANOMALY:", data)
        else:
            print("NORMAL:", data)

Consumer with ML started...
NORMAL: {'user_id': 44, 'amount': 345.43, 'timestamp': 1775123002.212343}
NORMAL: {'user_id': 20, 'amount': 458.08, 'timestamp': 1775123003.392444}
NORMAL: {'user_id': 66, 'amount': 323.72, 'timestamp': 1775123004.5725768}
NORMAL: {'user_id': 55, 'amount': 461.09, 'timestamp': 1775123005.7523522}
NORMAL: {'user_id': 19, 'amount': 147.04, 'timestamp': 1775123006.9333894}
NORMAL: {'user_id': 85, 'amount': 132.38, 'timestamp': 1775123008.113518}
NORMAL: {'user_id': 61, 'amount': 397.58, 'timestamp': 1775123009.2946239}
NORMAL: {'user_id': 72, 'amount': 173.21, 'timestamp': 1775123010.4745893}
NORMAL: {'user_id': 11, 'amount': 91.35, 'timestamp': 1775123011.6546125}
NORMAL: {'user_id': 11, 'amount': 369.49, 'timestamp': 1775123012.8361456}
NORMAL: {'user_id': 64, 'amount': 383.76, 'timestamp': 1775123014.016882}
NORMAL: {'user_id': 61, 'amount': 260.08, 'timestamp': 1775123015.1970344}
NORMAL: {'user_id': 70, 'amount': 141.59, 'timestamp': 1775123016.3771265}
NO